# PreRequisite

In [ ]:
import sys
import os

# Define the path to the directory containing the module
module_dir = "../Main/RequiredFuntions"

# Append the directory to sys.path
sys.path.append(module_dir)

# Initialization phase

In [ ]:
import InitializationPhase as IP

# GENERATING PUBLIC AND PRIVATE KEYS FOR USER
USER = IP.ECCPublicPrivateKeyGeneration()

# GENERATING PUBLIC AND PRIVATE KEYS FOR DEVICE
DEVICE = IP.ECCPublicPrivateKeyGeneration()

# GENERATING PUBLIC AND PRIVATE KEYS FOR GATEWAY
GATEWAY = IP.ECCPublicPrivateKeyGeneration()

In [ ]:
# GENERATING SYMMETRIC KEYS
## USER
SYMMETRIC_KEY_USER = IP.SymmetricKeyGeneration()

## DEVICE
SYMMETRIC_KEY_DEVICE = IP.SymmetricKeyGeneration()

## GATEWAY
SYMMETRIC_KEY_GATEWAY = IP.SymmetricKeyGeneration()

# USER - REGISTRATION PHASE

In [ ]:
import EncryptionDecryption as ED
import functions as fn
import dataTransfer as DT
import threading
import hashlib
import time

In [ ]:
# step 1
fingerprintImagePath = "../Main/RequiredData/Fingerprint/5.jpg"
accelerometerDataPath = "../Main/RequiredData/Accelerometer/m55s_training.csv"

In [ ]:
uniqueId_i = "RajeshDevice1"

imageHash = fn.hash_file(fingerprintImagePath)

accelerometerHash = fn.hash_file(accelerometerDataPath)

userNonce_i = fn.nonce_gen()

compute_i = hashlib.sha256((uniqueId_i 
                                + imageHash
                                + accelerometerHash
                                + str(userNonce_i)
                                + SYMMETRIC_KEY_GATEWAY
                                ).encode()
                            ).hexdigest()

encryptedIdandImage = ED.symmetric_key_encryption(uniqueId_i, 
                                                  ED.read_image_as_bytes(fingerprintImagePath), 
                                                  SYMMETRIC_KEY_GATEWAY)

encryptedIDandAccelerometerData = ED.symmetric_key_encryption(uniqueId_i,
                                                              ED.read_image_as_bytes(accelerometerDataPath), 
                                                              SYMMETRIC_KEY_GATEWAY)

In [ ]:
import imginfo

FingerImageMetaData = imginfo.extract_metadata(fingerprintImagePath)
CSVFileMetaData = imginfo.extract_metadata(accelerometerDataPath)

encryptedFingerImageMetaData = ED.encryption(FingerImageMetaData, GATEWAY[0])
encryptedCSVFileMetaData = ED.encryption(CSVFileMetaData, GATEWAY[0])

In [ ]:
userDeviceData = {
    "encryptedIdandImage" : encryptedIdandImage,
    "encryptedIDandAccelerometerData" : encryptedIDandAccelerometerData,
    "imageHash" : imageHash,
    "accelerometerHash" : accelerometerHash,
    "userNonce" : userNonce_i,
    "compute" : compute_i,
    "FingerImageMetaData" : encryptedFingerImageMetaData,
    "CSVFileMetaData" : encryptedCSVFileMetaData
}


In [ ]:
# Create and start threads
thread1 = threading.Thread(target=DT.receive)  # Start the receive function
thread2 = threading.Thread(target=DT.send, args=(userDeviceData,))  # Start the send function with arguments

thread1.start()
time.sleep(2)  # Ensure the server starts before sending
thread2.start()

# Wait for threads to complete
thread1.join()
thread2.join()

print("Data transfer and storage complete.")

In [ ]:
# step 2
print("Starting Step 2")
json_data = fn.read_json_file("../Main/ReceivedData/received_data.json")

In [ ]:
decryptedIdandImage = ED.symmetric_key_decryption(json_data["encryptedIdandImage"], 
                                                  SYMMETRIC_KEY_GATEWAY)

decryptedIDandAccelerometerData = ED.symmetric_key_decryption(json_data["encryptedIDandAccelerometerData"], 
                                                              SYMMETRIC_KEY_GATEWAY)

print(f"both the above decryptions gave the same output unique id : {decryptedIdandImage[0] == decryptedIDandAccelerometerData[0]}")

print("image hash validated : ", hashlib.sha256(decryptedIdandImage[1]).hexdigest() == json_data["imageHash"])

In [ ]:
validationCompute_i = hashlib.sha256((decryptedIdandImage[0].decode() + 
                                      hashlib.sha256(decryptedIdandImage[1]).hexdigest() + 
                                      hashlib.sha256(decryptedIDandAccelerometerData[1]).hexdigest() + 
                                      str(json_data["userNonce"]) +
                                      SYMMETRIC_KEY_GATEWAY).encode()).hexdigest()

In [ ]:
if(validationCompute_i == json_data["compute"]): print("validation for compute status : \"success\"")
else: print("validation \"failed\" ")

In [ ]:
baseAccelerometerData = "../Main/RequiredData/Accelerometer/baseTrainingData.csv"

import functions as fn
fn.AddNewDevice(1, "defaultUser1")

In [ ]:
# creating devicelist for default device for training (assuming already a device just for training purposes)
NewUser = fn.GetNewDeviceID(fn.load_all_data())
fn.AddNewDevice(NewUser, decryptedIdandImage[0].decode())

In [ ]:
import pandas as pd
import io

baseDF = pd.read_csv(baseAccelerometerData)

In [ ]:
df = pd.read_csv(io.StringIO(decryptedIDandAccelerometerData[1].decode('UTF-8')))
df = df.dropna()
df = df.drop(columns=["Var1_Timestamp", "Var2_Timestamp", "Var3_Timestamp"])
df["user"] = NewUser
df.head()

In [ ]:
Combined_training_data = pd.concat([baseDF, df], ignore_index=True)
Combined_training_data.to_csv(baseAccelerometerData, index=False)

In [ ]:
# import accelerometerTraining as AT
# TrainingThread = threading.Thread(target=AT.AccelerometerTraining, args=(Combined_training_data,))
# TrainingThread.start()

# print("waiting for 10sec......")
# time.sleep(10)

In [ ]:
# checking the liveness of the image

import joblib
import FingerBiometrics as FB

fingerprintModel = joblib.load('./Model/Fingerprint/naive_bayes_fingerprint_model.pkl')
result = FB.predict_fingerprint(decryptedIdandImage[1], (200,200), fingerprintModel)
print(f"The fingerprint is: {result}")

    #   should add a break statement if image is fake

In [ ]:
# isolate the finger from the image
import tensorflow as tf
import cv2
import numpy as np
import matplotlib.pyplot as plt

model = tf.saved_model.load('./Model/Fingerprint/ssd_mobilenet_v2_320x320_coco17_tpu-8/saved_model')
image_np = FB.load_image_into_numpy_array(decryptedIdandImage[1])
detections = FB.detect_objects(image_np, model)

boxes = detections['detection_boxes'][0].numpy()
scores = detections['detection_scores'][0].numpy()
classes = detections['detection_classes'][0].numpy().astype(np.int32)

cropped_image, image_with_boxes = FB.draw_best_box(image_np, boxes, scores, classes)

print("Detection and cropping completed.")

if image_with_boxes is not None:
    cropped_image_rgb = cv2.cvtColor(image_with_boxes, cv2.COLOR_BGR2RGB)

    # ✅ Display Image in Jupyter Notebook
    plt.figure(figsize=(6, 6))
    plt.imshow(cropped_image_rgb)
    plt.axis("off")  # Hide axes
    plt.title("Best Detected Fingerprint")
    plt.show()
else:
    print("❌ No valid cropped image found.")
